# 01 - Verificación del corpus normativo ambiental

Este notebook valida que el corpus documental inicial esté correctamente organizado antes de iniciar la extracción de texto, chunking, embeddings e indexación vectorial.

Verifica:

- existencia del archivo CSV de metadatos;
- existencia de la carpeta `data/raw/`;
- coincidencia entre la columna `archivo_pdf` y los PDFs reales;
- columnas obligatorias;
- IDs duplicados;
- valores permitidos en campos controlados;
- campos vacíos o no determinados;
- resumen general del corpus.


## 1. Importación de librerías

In [1]:
from pathlib import Path
import pandas as pd
import json
from datetime import datetime


## 2. Definición de rutas del proyecto

Este notebook está pensado para ejecutarse desde la raíz del repositorio o desde la carpeta `notebooks/`. Si hay problemas con las rutas, revisa el valor de `ROOT_DIR`.


In [2]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    ROOT_DIR = current_dir.parent
else:
    ROOT_DIR = current_dir

DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
METADATA_DIR = DATA_DIR / "metadata"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"

CSV_PATH = METADATA_DIR / "corpus_normativo_ambiental.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CSV_PATH: {CSV_PATH}")
print(f"RAW_DIR: {RAW_DIR}")


ROOT_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana
CSV_PATH: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\metadata\corpus_normativo_ambiental.csv
RAW_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\raw


## 3. Validación inicial de existencia de archivos y carpetas

In [30]:
errors = []
warnings = []

if not CSV_PATH.exists():
    errors.append(f"No se encontró el CSV de metadatos: {CSV_PATH}")

if not RAW_DIR.exists():
    errors.append(f"No se encontró la carpeta de PDFs: {RAW_DIR}")

if errors:
    print("ERRORES CRÍTICOS:")
    for err in errors:
        print(f"- {err}")
else:
    print("Estructura inicial encontrada correctamente.")


Estructura inicial encontrada correctamente.


## 4. Carga del CSV del corpus

In [31]:
df = pd.read_csv(CSV_PATH)

print(f"Cantidad de documentos registrados: {len(df)}")
df.head()


Cantidad de documentos registrados: 30


,id_documento,nombre_archivo_original,archivo_pdf,titulo_documento,tipo_norma,numero_norma,entidad_emisora,fecha_publicacion,tema_principal,subtema,fuente_oficial,url_fuente,texto_extraible,estado_vigencia,prioridad,observaciones
0,DOC-001,625.pdf,DOC-001_DS_004_2012_MINAM_IGAC.pdf,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,Minería y ambiente / IGAC,El Peruano,No determinado,Sí,No determinado,Alta,Norma relevante porque regula el IGAC aplicabl...
1,DOC-002,305-2017-MINAM.pdf,DOC-002_RM_305_2017_MINAM_CALIDAD_AIRE_GESTA.pdf,Aprueban los Lineamientos para el Fortalecimie...,Resolución Ministerial,Resolución Ministerial N.º 305-2017-MINAM,MINAM,2017-10-19,Calidad ambiental,ECA / Calidad del aire / GESTA,MINAM / El Peruano,No determinado,Parcial,No determinado,Alta,Norma relevante porque aprueba lineamientos pa...
2,DOC-003,656.pdf,DOC-003_RM_702_2008_MINSA_RESIDUOS_SEGREGADORE...,Aprueban la NTS N.º 073-2008-MINSA/DIGESA-V.01...,Resolución Ministerial,Resolución Ministerial N.º 702-2008/MINSA / NT...,MINSA / DIGESA,2008-10-09,Residuos sólidos,Residuos sólidos / Segregadores / Reaprovecham...,MINSA / DIGESA,http://www.minsa.gob.pe/portal/transparencia/n...,No,No determinado,Media,Documento oficial relevante porque establece u...
3,DOC-004,1466.pdf,DOC-004_LEY_28611_2005_CONGRESO_LEY_GENERAL_AM...,Ley General del Ambiente,Ley,Ley N.º 28611,Congreso de la República,2005-10-15,Gestión ambiental,SNGA / Gestión ambiental / Política ambiental,Congreso de la República / MINAM,https://www.leyes.congreso.gob.pe/documentos/l...,Sí,No determinado,Alta,Norma base del marco ambiental peruano. Establ...
4,DOC-005,1599.pdf,DOC-005_RM_554_2012_MINSA_RESIDUOS_ESTABLECIMI...,"Aprueban la NTS N.º 096-MINSA/DIGESA-V.01, Nor...",Resolución Ministerial,Resolución Ministerial N.º 554-2012/MINSA / NT...,MINSA / DIGESA,2012-07-03,Residuos sólidos,Residuos sólidos / Residuos hospitalarios / Ma...,MINSA / DIGESA,http://www.minsa.gob.pe/transparencia/dge_norm...,No,No determinado,Media,Documento oficial relevante porque regula la g...


## 5. Validación de columnas obligatorias

In [32]:
required_columns = [
    "id_documento",
    "nombre_archivo_original",
    "archivo_pdf",
    "titulo_documento",
    "tipo_norma",
    "numero_norma",
    "entidad_emisora",
    "fecha_publicacion",
    "tema_principal",
    "subtema",
    "fuente_oficial",
    "url_fuente",
    "texto_extraible",
    "estado_vigencia",
    "prioridad",
    "observaciones",
]

missing_columns = [col for col in required_columns if col not in df.columns]
extra_columns = [col for col in df.columns if col not in required_columns]

if missing_columns:
    errors.append(f"Faltan columnas obligatorias: {missing_columns}")
else:
    print("Todas las columnas obligatorias están presentes.")

if extra_columns:
    warnings.append(f"Hay columnas adicionales no previstas: {extra_columns}")
    print(f"Columnas adicionales detectadas: {extra_columns}")


Todas las columnas obligatorias están presentes.


## 6. Validación de IDs y duplicados

In [33]:
duplicated_ids = df[df["id_documento"].duplicated(keep=False)]["id_documento"].tolist()

if duplicated_ids:
    errors.append(f"Existen IDs duplicados: {duplicated_ids}")
else:
    print("No se encontraron IDs duplicados.")

invalid_id_format = df[~df["id_documento"].astype(str).str.match(r"^DOC-\d{3}$", na=False)]

if not invalid_id_format.empty:
    warnings.append("Algunos IDs no cumplen el formato DOC-XXX.")
    display(invalid_id_format[["id_documento", "titulo_documento"]])
else:
    print("Todos los IDs cumplen el formato DOC-XXX.")


No se encontraron IDs duplicados.
Todos los IDs cumplen el formato DOC-XXX.


## 7. Validación de existencia de PDFs en `data/raw/`

In [34]:
registered_files = set(df["archivo_pdf"].dropna().astype(str))
existing_files = set(p.name for p in RAW_DIR.glob("*.pdf"))

missing_pdfs = sorted(registered_files - existing_files)
unregistered_pdfs = sorted(existing_files - registered_files)

if missing_pdfs:
    errors.append(f"PDFs registrados en el CSV que no existen en data/raw: {missing_pdfs}")
    print("PDFs faltantes:")
    for f in missing_pdfs:
        print(f"- {f}")
else:
    print("Todos los PDFs registrados existen en data/raw.")

if unregistered_pdfs:
    warnings.append(f"Hay PDFs en data/raw no registrados en el CSV: {unregistered_pdfs}")
    print("PDFs no registrados en el CSV:")
    for f in unregistered_pdfs:
        print(f"- {f}")
else:
    print("No hay PDFs adicionales sin registrar.")


Todos los PDFs registrados existen en data/raw.
No hay PDFs adicionales sin registrar.


## 8. Validación de campos controlados

Se revisa que los campos tengan valores consistentes.


In [35]:
allowed_values = {
    "texto_extraible": {"Sí", "Si", "No", "Parcial", "No determinado"},
    "estado_vigencia": {"Vigente", "Modificado", "Derogado", "No determinado"},
    "prioridad": {"Alta", "Media", "Baja", "No determinado"},
}

for column, allowed in allowed_values.items():
    invalid_rows = df[~df[column].fillna("No determinado").astype(str).isin(allowed)]
    if not invalid_rows.empty:
        warnings.append(f"Valores no permitidos en {column}.")
        print(f"Valores no permitidos en {column}:")
        display(invalid_rows[["id_documento", column, "titulo_documento"]])
    else:
        print(f"Campo {column}: valores correctos.")


Campo texto_extraible: valores correctos.
Campo estado_vigencia: valores correctos.
Campo prioridad: valores correctos.


## 9. Revisión de campos vacíos o no determinados

In [36]:
fields_to_review = [
    "titulo_documento",
    "tipo_norma",
    "numero_norma",
    "entidad_emisora",
    "fecha_publicacion",
    "tema_principal",
    "subtema",
    "fuente_oficial",
    "url_fuente",
    "estado_vigencia",
]

review_summary = []

for field in fields_to_review:
    empty_count = df[field].isna().sum()
    not_determined_count = (df[field].astype(str).str.strip() == "No determinado").sum()
    review_summary.append({
        "campo": field,
        "vacios": int(empty_count),
        "no_determinado": int(not_determined_count),
    })

review_df = pd.DataFrame(review_summary)
review_df


,campo,vacios,no_determinado
0,titulo_documento,0,0
1,tipo_norma,0,0
2,numero_norma,0,1
3,entidad_emisora,0,0
4,fecha_publicacion,0,2
5,tema_principal,0,0
6,subtema,0,0
7,fuente_oficial,0,0
8,url_fuente,0,26
9,estado_vigencia,0,30


## 10. Resumen del corpus

In [37]:
print("Resumen por tema principal:")
display(df["tema_principal"].value_counts().reset_index().rename(columns={"index": "tema_principal", "tema_principal": "cantidad"}))

print("Resumen por tipo de norma:")
display(df["tipo_norma"].value_counts().reset_index().rename(columns={"index": "tipo_norma", "tipo_norma": "cantidad"}))

print("Resumen por prioridad:")
display(df["prioridad"].value_counts().reset_index().rename(columns={"index": "prioridad", "prioridad": "cantidad"}))


Resumen por tema principal:


,cantidad,count
0,Calidad ambiental,17
1,Gestión ambiental,6
2,Evaluación ambiental,3
3,Residuos sólidos,2
4,Biodiversidad,1
5,Fiscalización ambiental,1


Resumen por tipo de norma:


,cantidad,count
0,Decreto Supremo,14
1,Resolución Ministerial,8
2,Otro,5
3,Ley,2
4,Norma técnica,1


Resumen por prioridad:


,cantidad,count
0,Alta,17
1,Media,9
2,Baja,4


## 11. Generación de reporte de verificación

El reporte se guardará en `experiments/resultados/`.


In [38]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

report = {
    "fecha_verificacion": datetime.now().isoformat(timespec="seconds"),
    "total_documentos_csv": int(len(df)),
    "total_pdfs_raw": int(len(existing_files)),
    "pdfs_faltantes": missing_pdfs,
    "pdfs_no_registrados": unregistered_pdfs,
    "errores": errors,
    "advertencias": warnings,
}

report_path = REPORTS_DIR / f"reporte_verificacion_corpus_{timestamp}.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

review_csv_path = REPORTS_DIR / f"resumen_campos_corpus_{timestamp}.csv"
review_df.to_csv(review_csv_path, index=False, encoding="utf-8-sig")

print(f"Reporte JSON guardado en: {report_path}")
print(f"Resumen CSV guardado en: {review_csv_path}")


Reporte JSON guardado en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\experiments\resultados\reporte_verificacion_corpus_20260519_204934.json
Resumen CSV guardado en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\experiments\resultados\resumen_campos_corpus_20260519_204934.csv


## 12. Resultado final de la verificación

In [39]:
if errors:
    print("La verificación encontró errores que deben corregirse antes de continuar:")
    for err in errors:
        print(f"- {err}")
else:
    print("El corpus está listo para pasar a la etapa de extracción de texto.")

if warnings:
    print("\nAdvertencias a revisar:")
    for warn in warnings:
        print(f"- {warn}")


El corpus está listo para pasar a la etapa de extracción de texto.
